# Imports

In [ ]:
import time
import ipywidgets as widgets

from pathlib import Path
from ipywidgets import interact

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Functions

Here are the functions used throughout the notebook.

## Plotting

In [ ]:
def plot_signal(data: pd.DataFrame) -> tuple[plt.figure, plt.axes]:
    fig, ax = plt.subplots(figsize=(15,8))
    ax.plot(data)

    ax.grid(visible=True)
    ax.tick_params(axis='both', labelsize=14)

    ax.set_xlabel("time (a.u.)", fontsize=18)
    ax.set_ylabel("amplitude ($V$)", fontsize=18)
    
    return fig, ax


def plot_multipeaks(signal: pd.DataFrame, peak_idx: pd.Series, props: dict):
    """
    signal: `pd.DataFrame` or `pd.Series` containing the values of the signal
    peak_idx: Index of all the peaks; first output of `scipy.signal.findpeaks()`
    props: Metadata on the peaks; second output of `scipy.signal.find_peaks()`
    """
    fig, ax = plot_signal(signal)
    
    peak_heights = props[signal.name]['peak_heights']
    peak_locs = peak_idx[signal.name]
    
    ax.plot(peak_locs, peak_heights, 'rx')
    ax.set_title(f"Plot of Signal {signal.name}")

## RMS

In [ ]:
def rms(series: pd.Series, index: int) -> pd.Series:
    n = len(series.values[0:index])
    total = sum(series.values[0:index] ** 2)
    return np.sqrt(1/n * total)


def subtract_rms(series: pd.Series, baseline_rms: pd.Series) -> pd.Series:
    return series - baseline_rms[series.name]

## SNR

In [ ]:
def snr_filter(peak_height: float, baseline_rms: float, threshold=1):
    return False if peak_height / baseline_rms < threshold else True

## Multipeaks

In [ ]:
def is_single(series):
    return True if len(series) == 1 else False

# Preliminary Cleaning

Here we remove load the data and immediately remove any data with missing, $\mathrm{NaN}$, or $\inf$ values. We then remove the set DC offset and invert the signals for better visualization. 

**Note:** To begin, please select a file by specifing the path of `PARQ_PATH` in the following cell.

In [ ]:
# Prep Steps
PARQ_PATH = Path("../sample_datasets/20220824_CERC_background/raw_data/parquet/20220824-0003.parquet")

In [ ]:
# Loading Steps
start = time.perf_counter()
df = pd.read_parquet(PARQ_PATH)

df.columns = df.columns.astype('int16')

load_parquet_time = time.perf_counter() - start
num_initial_signals = df.shape[0]

print(
    f"\x1b[1;36m{num_initial_signals}\x1b[0m signals loaded in "
    f"\x1b[1;32m{(load_parquet_time)*1000:.2f} ms\x1b[0m."
)

In [ ]:
# Drop any signals that have missing or infinite values
start = time.perf_counter()

df = df.replace([np.inf, -np.inf], np.nan).dropna(how='any')

remove_missing_time = time.perf_counter() - start
num_missing_signals = num_initial_signals - df.shape[0]

print(
    f"Removed \x1b[1;31m{num_missing_signals}\x1b[0m signals "
    f"with missing or infinite values. "
    f"[\x1b[1;32m{(remove_missing_time)*1000:.2f} ms\x1b[0m]"
)

In [ ]:
df = df.T
df.head()

In [ ]:
plot_signal(df)
plt.show()

In [ ]:
# Adjusting Steps

DC_OFFSET = 0.95  # Volts
df = df - DC_OFFSET
df = df * -1

fig, ax = plot_signal(df)

# Baseline RMS Subtraction

Here we remove the baseline root-mean-squared (RMS) value following the data processing outlined in Section 3.2 from [this](https://www1.grc.nasa.gov/wp-content/uploads/TM-20205008493-Fast-Neutron-Spectroscopy-with-Organic-Scintillation-Detectors-in-a-High-Radiation-Field.pdf) paper by NASA.

In [ ]:
baseline_cutoff = 64  # Index of time (64ns)

In [ ]:
fig, ax = plot_signal(df)
ax.plot(baseline_cutoff, 0)
ax.vlines(baseline_cutoff, -0.2, 3, linestyles="dashed")
ax.set_ylim(-0.175, 2)
ax.set_title(f"Signals with Baseline Cutoff of {baseline_cutoff} Highlighted")

plt.show()

In [ ]:
baseline_rms = df.apply(lambda x: rms(x, baseline_cutoff))

In [ ]:
df_offset = df.apply(lambda x: subtract_rms(x, baseline_rms))

In [ ]:
fig, ax = plot_signal(df_offset)
ax.set_title("Signals with Baseline RMS Removed")
plt.show()

# Multipeak Removal

Here we removed any signals that have any number of peaks that is not equal to 1. This is to ensure that we do not have overlapping signals and/or false positives. Arguments `height` and `prominence` in [this](#peak-cell) cell be adjusted to create a finer filter.

In [ ]:
import scipy.signal as signal

<a id="peak-cell"></a>

In [ ]:
output = df_offset.apply(lambda x: signal.find_peaks(x.values, height=0.01, prominence=0.1))
peak_idx, props = output.iloc[0,:], output.iloc[1,:]

In [ ]:
filt = peak_idx.apply(is_single)

In [ ]:
initial_size = df_offset.shape[1]

df_single_peaks = df_offset.T[filt.values]
df_single_peaks = df_single_peaks.T

final_size = df_single_peaks.shape[1]

print(
    f"Removed \x1b[1;31m{initial_size - final_size}\x1b[0m multipeaked signals, "
    f"\x1b[1;32m{final_size}\x1b[0m untouched."
)

In [ ]:
fig, ax = plot_signal(df_single_peaks)
ax.set_title("Signals with Single Peaks")
plt.show()

## Visualize Multipeak Signals

Use the dropdown menu to visualize the signals which were indentified to have multiple peaks. Peaks are highlighted in red.

If the results are undesirable, try changing `height` and/or `prominence` values when using `is_single()` to try and refine the results.

In [ ]:
df_multipeaks = df_offset.T[~filt.values].T
bad_signal_ids = df_multipeaks.columns

In [ ]:
interact(
    lambda signal_id: plot_multipeaks(df_multipeaks[signal_id], peak_idx, props), 
    signal_id=bad_signal_ids
)

plt.show()

# Incomplete Triggers?

In [ ]:
# Unsure here

# SNR Filtering

Here we remove any signals with low SNR values defined as:

$\text{SNR} = \dfrac{\text{Peak Height}}{\text{Baseline } \text{RMS}} < \text{threshold}$

Adjust the argument `threshold` in [functions](#SNR) to create finer filters.

In [ ]:
df_single_peaks

In [ ]:
initial_size = df_single_peaks.shape[1]

filt = df_single_peaks.apply(
    lambda x: snr_filter(props[x.name]['peak_heights'], baseline_rms[x.name], threshold=1)
)

df_cleaned = df_single_peaks.T[filt.values].T

final_size = df_cleaned.shape[1]

print(
    f"Removed \x1b[1;31m{initial_size - final_size}\x1b[0m signals with low signal-to-noise ratio, "
    f"\x1b[1;32m{final_size}\x1b[0m untouched."
)

## Visualize Bad Signals

In [ ]:
df_bad_snr = df_single_peaks.T[~filt.values].T

# Appendix: Time Analysis